# Benchmark M1–M6 — Échantillon (Configuration B1 : LLM seul, zero-shot)

**Modèle testé :** Gemma 4 31B (via vLLM, API OpenAI-compatible)  
**Modules couverts :** M1 (Principe QA), M2 (Applied QA), M4 (QCM), M6 (Interprétation d'arrêt)  
**Modules skippés :** M3 (multi-saut, nécessite KG v1), M5 (temporalité, nécessite versioning)  

---
**Prérequis :**
1. Serveur vLLM démarré via `setup_cluster.sh`
2. Dossier `cluster_data/` présent (généré par `prepare_cluster_data.py` en local)
3. Ajuster les variables de la cellule **Configuration** ci-dessous

---

## 0. Setup cluster — install, download, démarrage vLLM

**À exécuter une fois par session.** Ces 6 cellules préparent l'environnement directement depuis le notebook, sans repasser par un script bash :

| #   | Cellule                        | Ce qu'elle fait                                      |
| --- | ------------------------------ | ---------------------------------------------------- |
| 0.2 | Config cluster                 | Détecte les GPU, fixe le modèle et le port           |
| 0.3 | Installation des dépendances   | `%pip install` vllm + clients dans le venv du kernel |
| 0.4 | Auth HF + download             | Téléchargement du modèle (30-60 min la 1re fois)     |
| 0.5 | Démarrage vLLM                 | `subprocess.Popen` en arrière-plan, logs dans `./logs/` |
| 0.6 | Health check                   | Attend que `/health` réponde 200 (≈ 5-10 min)        |

> **Pré-requis** : le notebook doit tourner dans un venv (bootstrap via `bash setup_cluster.sh` → crée `~/.venv-benchmark` + lance jupyter dedans).
> Vérification rapide : `!which python` doit pointer sur `~/.venv-benchmark/bin/python`.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 0.2 — CONFIG CLUSTER (à adapter selon le node)
# ═══════════════════════════════════════════════════════════════════════
import os
import subprocess
from pathlib import Path

# ── Modèle ────────────────────────────────────────────────────────────
# Gemma 4 E4B-it : ~4.5B params effectifs, ~16 GB VRAM en bf16
# Tient largement sur 1×A40 (44 GB) avec marge pour kv-cache
# Alternative qualité supérieure : "cyankiwi/gemma-4-26B-A4B-it-AWQ-4bit"
MODEL_ID  = "google/gemma-4-E4B-it"

# ── Serveur vLLM ──────────────────────────────────────────────────────
VLLM_PORT = 8000
MAX_LEN   = 8192        # longueur de contexte
GPU_UTIL  = 0.90        # fraction VRAM allouée à vLLM

# ── Auth HuggingFace (modèles gated) ──────────────────────────────────
# Option 1 : export HF_TOKEN=hf_xxxx avant de lancer jupyter
# Option 2 : écrire le token ici (⚠️ NE PAS committer)
HF_TOKEN = os.environ.get("HF_TOKEN")   # ou "hf_xxx"

# ── Détection GPU (pour le tensor parallelism) ────────────────────────
try:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True,
    )
    gpus = [l for l in out.strip().split("\n") if l]
    NUM_GPUS = len(gpus)
    print(f"GPUs détectés : {NUM_GPUS}")
    for i, g in enumerate(gpus):
        print(f"  [{i}] {g}")
except Exception as e:
    print(f"[WARN] nvidia-smi indisponible ({e}) → NUM_GPUS=1")
    NUM_GPUS = 1

# ── Chemins ────────────────────────────────────────────────────────────
LOG_DIR = Path("./logs"); LOG_DIR.mkdir(exist_ok=True)
VLLM_LOG = LOG_DIR / "vllm_server.log"
VLLM_PID = LOG_DIR / "vllm.pid"

# ── URL dérivée (utilisée par les cellules suivantes + le benchmark) ──
VLLM_BASE_URL = f"http://localhost:{VLLM_PORT}/v1"
print(f"\nModèle        : {MODEL_ID}")
print(f"vLLM serveur  : {VLLM_BASE_URL}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 0.3 — INSTALLATION DES DÉPENDANCES
# %pip installe dans le venv du kernel courant (évite les erreurs PEP 668).
# ═══════════════════════════════════════════════════════════════════════

%pip install -q --upgrade pip
%pip install -q "vllm>=0.8.5"
%pip install -q "openai>=1.50" "pydantic>=2" "pandas>=2" "pyarrow>=14" \
                "huggingface_hub>=0.25" "datasets>=3" "rich>=13" \
                "matplotlib>=3.8" "seaborn>=0.13" "tqdm>=4.65"

print("\n✓ Dépendances installées")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 0.4 — AUTH HUGGINGFACE + TÉLÉCHARGEMENT DU MODÈLE
# ═══════════════════════════════════════════════════════════════════════
from huggingface_hub import login, snapshot_download

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✓ Authentifié sur HuggingFace")
else:
    print("[WARN] HF_TOKEN non défini — si le modèle est gated, le download va échouer.")
    print("       Solution : exporter HF_TOKEN=hf_xxx avant de lancer jupyter,")
    print("       puis redémarrer le kernel (Kernel > Restart).")

print(f"\nTéléchargement de {MODEL_ID}…")
print("(première fois : 30-60 min selon bande passante — les fois suivantes : cache HF réutilisé)")
model_path = snapshot_download(
    repo_id=MODEL_ID,
    ignore_patterns=["*.md", "*.txt", "original/*"],
)
print(f"\n✓ Modèle en cache : {model_path}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 0.5 — DÉMARRAGE DU SERVEUR vLLM (en arrière-plan)
# ═══════════════════════════════════════════════════════════════════════
import subprocess
import os
import signal
import time

# Arrêt propre d'un éventuel serveur lancé dans une session précédente
if VLLM_PID.exists():
    try:
        old = int(VLLM_PID.read_text())
        os.killpg(os.getpgid(old), signal.SIGTERM)
        print(f"Serveur précédent (PID={old}) arrêté")
        time.sleep(3)
    except (ProcessLookupError, ValueError, PermissionError):
        pass

cmd = [
    "vllm", "serve", MODEL_ID,
    "--tensor-parallel-size", str(NUM_GPUS),
    "--max-model-len", str(MAX_LEN),
    "--gpu-memory-utilization", str(GPU_UTIL),
    "--port", str(VLLM_PORT),
]
print("Commande :", " ".join(cmd))

log_f = open(VLLM_LOG, "w")
vllm_proc = subprocess.Popen(
    cmd,
    stdout=log_f,
    stderr=subprocess.STDOUT,
    preexec_fn=os.setsid,       # nouveau groupe de process → killpg possible
)
VLLM_PID.write_text(str(vllm_proc.pid))
print(f"✓ vLLM démarré (PID={vllm_proc.pid})  ·  logs → {VLLM_LOG}")
print("  (le chargement du modèle en VRAM prend ~3-5 min pour le 26B-A4B)")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 0.6 — ATTENTE DU DÉMARRAGE (chargement du modèle en VRAM)
# ═══════════════════════════════════════════════════════════════════════
import time
import urllib.request

HEALTH_URL = f"http://localhost:{VLLM_PORT}/health"
MAX_WAIT_S = 900   # 15 min max (le 27B met ~5-10 min à charger en VRAM)

t0 = time.time()
ready = False
while time.time() - t0 < MAX_WAIT_S:
    # Si vLLM a crashé, pas la peine d'attendre
    if vllm_proc.poll() is not None:
        raise RuntimeError(
            f"vLLM s'est arrêté (code={vllm_proc.returncode}) — inspecter {VLLM_LOG}"
        )
    try:
        with urllib.request.urlopen(HEALTH_URL, timeout=2) as r:
            if r.status == 200:
                ready = True
                break
    except Exception:
        pass
    elapsed = int(time.time() - t0)
    print(f"  chargement… {elapsed}s (max {MAX_WAIT_S}s)", end="\r")
    time.sleep(5)

if not ready:
    raise RuntimeError(f"vLLM non prêt après {MAX_WAIT_S}s — voir {VLLM_LOG}")

print(f"\n✓ vLLM prêt sur {VLLM_BASE_URL}  (démarrage : {int(time.time()-t0)}s)")

# Afficher les modèles exposés par le serveur
import json
with urllib.request.urlopen(f"http://localhost:{VLLM_PORT}/v1/models") as r:
    data = json.load(r)
print("\nModèles disponibles :")
for m in data.get("data", []):
    print(f"  - {m['id']}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CONFIGURATION BENCHMARK
# Les variables cluster (MODEL_ID, VLLM_BASE_URL, NUM_GPUS) sont déjà
# définies dans la section 0 (Setup cluster).
# ═══════════════════════════════════════════════════════════════════════

API_KEY       = "EMPTY"                 # vLLM n'impose pas de clé

DATA_DIR      = "./cluster_data"        # Dossier des données pré-exportées
RESULTS_DIR   = "./results"             # Répertoire de sortie

# Paramètres LLM
TEMPERATURE   = 0.0        # Déterministe pour reproductibilité

# Budgets de tokens (générés par le LLM) — dimensionnés pour éviter
# la troncature du JSON sur des réponses juridiques étoffées en français.
# ≈ 3-4 chars/token → 1024 tokens ≈ 3500 chars ≈ 4-5 paragraphes.
MAX_TOKENS_M1 = 1536       # ↑ bumped : évite les coupures (voir tokens_used=3000 sur runs précédents)
MAX_TOKENS_M2 = 1536       # + risques + stratégie
MAX_TOKENS_M4 = 512        # QCM (choix + justif) — reste compact
MAX_TOKENS_M6 = 2048       # 7 dimensions incluant dispositif_summary

# Troncature du texte de décision pour M6 (caractères)
MAX_DECISION_CHARS = 4000

# ── M1 : échantillonnage sur le fichier complet (~2670 cas) ───────────
# Le fichier cluster_data/m1_full.json contient TOUTES les questions de
# Les-Audits-Affaires. On en tire un sous-échantillon aléatoire à chaque
# run pour contrôler le coût/durée.
#
#   M1_N_QUESTIONS = 10   → test rapide (~2 min)
#   M1_N_QUESTIONS = 500  → run statistique (~1-2 h selon modèle)
#   M1_N_QUESTIONS = 2670 → run complet
#
# Fixer M1_SEED garantit la reproductibilité : deux runs avec la même
# graine tomberont sur les MÊMES questions → comparaison B1 vs B2 valide.
M1_N_QUESTIONS = 500
M1_SEED        = 42


In [ ]:
import json
import os
import time
import re
from pathlib import Path
from typing import Any, Literal, Optional
from datetime import datetime

import pandas as pd
from pydantic import BaseModel, Field, field_validator
from openai import OpenAI

# Afficher les cellules texte en plein (pas de troncature à 50 chars)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    import seaborn as sns
    MATPLOTLIB_OK = True
    plt.rcParams['figure.dpi'] = 120
    sns.set_theme(style="whitegrid", palette="muted")
except ImportError:
    MATPLOTLIB_OK = False
    print("[WARN] matplotlib/seaborn non disponibles — graphiques désactivés")

Path(RESULTS_DIR).mkdir(exist_ok=True)

# ── Timestamp du run ──────────────────────────────────────────────────
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Run ID : {RUN_ID}")
print(f"Résultats dans : {RESULTS_DIR}/")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# SCHÉMAS PYDANTIC (définis inline — notebook auto-contenu)
# ═══════════════════════════════════════════════════════════════════════

class M1Output(BaseModel):
    """Sortie attendue pour Module 1 — Principe QA."""
    reponse: str = Field(description="Réponse juridique complète en 2-4 paragraphes")
    articles_cites: list[str] = Field(description="Articles de loi cités (ex: 'art. 1641 C. civ.')")
    jurisprudence_citee: list[str] = Field(description="Arrêts cités (ex: 'Cass. soc., 15 janv. 2020')")
    niveau_certitude: Literal["élevé", "moyen", "faible"] = Field(description="Niveau de certitude de la réponse")


class M2Output(BaseModel):
    """Sortie attendue pour Module 2 — Applied QA (avec contexte client)."""
    reponse: str = Field(description="Analyse juridique spécifique à la situation")
    articles_cites: list[str] = Field(description="Articles de loi cités")
    jurisprudence_citee: list[str] = Field(description="Arrêts cités")
    risques: str = Field(description="Risques juridiques identifiés pour le client")
    strategie: str = Field(description="Recommandation pratique (action à mener)")


class M4Output(BaseModel):
    """Sortie attendue pour Module 4 — QCM retrieval."""
    reponse: Literal["A", "B", "C", "D"] = Field(description="Lettre de la bonne réponse")
    justification: str = Field(description="Explication juridique du choix")


class M6Output(BaseModel):
    """Sortie attendue pour Module 6 — Interprétation d'arrêt (7 dimensions)."""
    camp_in_decision: str = Field(description="Qui représente la position du client dans cette décision ?")
    sens_arret: Literal[
        "cassation", "cassation_partielle", "rejet",
        "confirmation", "infirmation", "infirmation_partielle",
        "accueil", "deboute", "non_lieu", "autre"
    ] = Field(description="Dispositif global de la décision")
    is_favorable: bool = Field(description="La décision est-elle favorable au client actuel ?")
    dispositif_summary: str = Field(description="Résumé de ce que décide la cour (2-4 phrases)")
    relevance: float = Field(ge=0.0, le=1.0, description="Score de pertinence pour le dossier (0=inutile, 1=décisif)")
    principles_extracted: list[str] = Field(description="Principes de droit utiles pour le dossier (liste)")
    transfer_reasoning: str = Field(description="Comment cette décision s'applique au dossier du client (1 paragraphe)")


print("Schémas Pydantic chargés.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# VÉRIFICATION DU SERVEUR VLLM
# ═══════════════════════════════════════════════════════════════════════

client = OpenAI(base_url=VLLM_BASE_URL, api_key=API_KEY)

try:
    models = client.models.list()
    model_ids = [m.id for m in models.data]
    print(f"Serveur vLLM OK — modèles disponibles : {model_ids}")

    # Vérifier que le modèle configuré est présent
    if MODEL_ID not in model_ids:
        print(f"[WARN] MODEL_ID='{MODEL_ID}' absent du serveur.")
        print(f"       Modèles disponibles : {model_ids}")
        print(f"       Mise à jour de MODEL_ID → '{model_ids[0]}'")
        MODEL_ID = model_ids[0]

    print(f"\nModèle actif : {MODEL_ID}")

    # Test rapide
    t0 = time.time()
    resp = client.chat.completions.create(
        model=MODEL_ID,
        messages=[{"role": "user", "content": "Répondez uniquement : OK"}],
        max_tokens=10,
        temperature=0.0,
    )
    latency = time.time() - t0
    print(f"Test LLM → '{resp.choices[0].message.content.strip()}' ({latency:.2f}s)")

except Exception as e:
    print(f"[ERREUR] Impossible de joindre le serveur vLLM : {e}")
    print(f"         Vérifiez que setup_cluster.sh a bien démarré le serveur sur {VLLM_BASE_URL}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# FONCTIONS UTILITAIRES
# ═══════════════════════════════════════════════════════════════════════

def call_llm_structured(
    system_prompt: str,
    user_prompt: str,
    output_model: type[BaseModel],
    max_tokens: int = 512,
) -> tuple[BaseModel | None, dict]:
    """
    Appel vLLM avec sortie JSON structurée (OpenAI response_format).
    Retourne (output_pydantic, meta). meta["raw_response"] contient la
    sortie brute du LLM même en cas d'échec du parse Pydantic.
    """
    t0 = time.time()
    meta = {
        "latency_s": None, "tokens_used": None, "finish_reason": None,
        "error": None, "raw_response": None,
    }

    try:
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt},
            ],
            temperature=TEMPERATURE,
            max_tokens=max_tokens,
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": output_model.__name__,
                    "schema": output_model.model_json_schema(),
                    "strict": True,
                },
            },
        )
        raw = response.choices[0].message.content
        meta["latency_s"]     = round(time.time() - t0, 2)
        meta["tokens_used"]   = response.usage.completion_tokens if response.usage else None
        meta["finish_reason"] = response.choices[0].finish_reason
        meta["raw_response"]  = raw

        if meta["finish_reason"] == "length":
            meta["error"] = (
                f"Coupé par max_tokens={max_tokens} (JSON probablement tronqué). "
                f"Augmenter le budget pour ce module."
            )
            return None, meta

        json_match = re.search(r'\{.*\}', raw, re.DOTALL)
        json_str = json_match.group(0) if json_match else raw
        result = output_model.model_validate_json(json_str)
        return result, meta

    except Exception as e:
        meta["latency_s"] = round(time.time() - t0, 2)
        meta["error"] = str(e)
        return None, meta


# ─────────────────────────────────────────────────────────────────────
# Normalisation des références d'articles
# ─────────────────────────────────────────────────────────────────────
# Le gold (extrait automatiquement par regex de LegMLAI) est au format :
#   "article 1641"              → 1641
#   "article L1121-1"           → L1121-1 (L collé)
#   "article L. 1121-1"         → L1121-1 (L. séparé)
#   "article 302 septies A"     → 302 septies A (suffixe latin)
#   "article 80 ter"            → 80 ter
#
# Le LLM peut répondre dans des formats variés :
#   "art. L. 1121-1 du Code du travail"
#   "article 1641 C. civ."
#   "L.1121-1"
#
# Le normaliseur les ramène tous à une clé canonique comparable.

def _article_key(s: str) -> str:
    """Ramène une référence d'article à une clé canonique pour comparaison.

    Ex : 'art. L. 1121-1 du Code du travail' → 'l1121-1'
         'article L1121-1'                    → 'l1121-1'
         'article 302 septies A'              → '302septiesa'
    """
    if not s:
        return ""
    s = s.lower().strip()
    # Retire le préfixe "article" ou "art."
    s = re.sub(r'^\s*(article|art\.?)\s+', '', s)
    # Retire "... du Code du travail", "C. civ.", "C. com.", etc. (tout ce qui suit le n°)
    s = re.sub(r'\s+(du|de la|des)\s+code.*$', '', s)
    s = re.sub(r'\s+c\.\s*(civ|com|trav|pén|pen|pr\.?\s*pén|proc|gén|ccp)\.?.*$', '', s)
    # Retire numéro romain suffixe (ex: "L. 442-1, II")
    s = re.sub(r',?\s+[ivx]+(\s|$)', ' ', s)
    # Normalise "L. 1121-1" ou "L 1121-1" → "l1121-1" (L collé au numéro)
    s = re.sub(r'\bl\.?\s*(?=\d)', 'l', s)
    # Compacte espaces et points, normalise tirets
    s = re.sub(r'[\s\.]+', '', s)
    s = s.replace('\u2013', '-').replace('\u2014', '-')
    return s


def article_recall(cited: list[str], gold: list[str]) -> float:
    """Fraction des articles gold retrouvés dans les articles cités.

    Compare via clés normalisées (`_article_key`) → tolère les variations
    de format ('L. 1121-1' ≡ 'L1121-1' ≡ 'L.1121-1').
    Match direct (égalité) ou par inclusion (gold ⊂ cited ou l'inverse).
    """
    if not gold:
        return 1.0  # pas de gold → pas de pénalité
    gold_keys  = {_article_key(g) for g in gold if g}
    cited_keys = {_article_key(c) for c in cited if c}
    gold_keys.discard("")
    cited_keys.discard("")
    if not gold_keys:
        return 1.0
    if not cited_keys:
        return 0.0
    found = 0
    for gk in gold_keys:
        if any(gk == ck or (len(gk) >= 3 and (gk in ck or ck in gk))
               for ck in cited_keys):
            found += 1
    return found / len(gold_keys)


def keyword_recall(text: str, keywords: list[str]) -> float:
    """Fraction des mots-clés gold trouvés dans le texte de réponse."""
    if not keywords:
        return 0.0
    text_lower = text.lower()
    found = sum(1 for kw in keywords if kw.lower() in text_lower)
    return found / len(keywords)


def principles_keyword_overlap(predicted: list[str], gold: list[str]) -> float:
    """Overlap des mots-clés entre les principes prédits et gold (M6)."""
    if not gold:
        return 0.0
    stopwords = {"le", "la", "les", "de", "du", "des", "en", "et", "ou",
                 "un", "une", "il", "est", "que", "qui", "sur", "par",
                 "à", "au", "avec", "pour", "pas", "ne", "se"}

    def extract_kw(texts):
        words = set()
        for t in texts:
            for w in re.findall(r'[a-zA-Zéèêëàâùûüçœî]{4,}', t.lower()):
                if w not in stopwords:
                    words.add(w)
        return words

    pred_kw = extract_kw(predicted)
    gold_kw = extract_kw(gold)
    if not pred_kw and not gold_kw:
        return 0.0
    return len(pred_kw & gold_kw) / len(pred_kw | gold_kw)


def format_decision_for_prompt(decision: dict, max_chars: int = MAX_DECISION_CHARS) -> str:
    """Formate le texte d'une décision pour le prompt M6 (tronqué)."""
    s = decision.get("structure") or {}
    parts = []
    header = (
        f"Juridiction : {decision.get('juridiction', '?')} — {decision.get('chambre', '')}\n"
        f"Date : {decision.get('date', '?')} | N° : {decision.get('numero', '?')}\n"
    )
    parts.append(header)
    if s.get("faits"):
        parts.append(f"[FAITS]\n{s['faits'][:600]}")
    if s.get("motifs"):
        parts.append(f"[MOTIFS]\n{s['motifs'][:2200]}")
    if s.get("dispositif"):
        parts.append(f"[DISPOSITIF]\n{s['dispositif'][:600]}")
    if not parts[1:]:
        raw = decision.get("full_text", "")
        parts.append(raw[:max_chars])
    text = "\n\n".join(parts)
    if len(text) > max_chars:
        text = text[:max_chars] + "\n[...texte tronqué...]"
    return text


def print_full_details(df: pd.DataFrame, sections, title: str = "Détails complets") -> None:
    """Imprime chaque ligne du df avec les champs demandés en plein texte."""
    if df.empty:
        print("(aucun résultat)")
        return
    print("\n" + "═" * 80)
    print(f"  {title}  ({len(df)} cas)")
    print("═" * 80)
    for _, row in df.iterrows():
        header = f"  {row.get('id', '?')}"
        extras = []
        for k in ("specialisation", "difficulty"):
            if k in row.index and pd.notna(row[k]):
                extras.append(str(row[k]))
        if extras:
            header += "  —  " + "  ·  ".join(extras)
        print("\n" + "─" * 80)
        print(header)
        print("─" * 80)
        for sec in sections:
            label, field = (sec, sec) if isinstance(sec, str) else sec
            if field not in row.index:
                continue
            val = row[field]
            if val is None:
                continue
            if isinstance(val, float) and pd.isna(val):
                continue
            print(f"\n▸ {label}")
            if isinstance(val, (list, tuple)):
                if not val:
                    print("  (vide)")
                else:
                    for item in val:
                        print(f"  • {item}")
            elif isinstance(val, bool):
                print(f"  {val}")
            else:
                for line in str(val).split("\n"):
                    print(f"  {line}")


# ─────────────────────────────────────────────────────────────────────
# Sanity check du normaliseur (s'exécute à l'import de la cellule)
# ─────────────────────────────────────────────────────────────────────
_sanity_pairs = [
    ("article L1121-1",                        "art. L. 1121-1 du Code du travail"),
    ("article 1641",                           "art. 1641 C. civ."),
    ("article 302 septies A",                  "article 302 septies A du CGI"),
    ("article L. 442-1, II du Code de commerce", "art. L442-1 c. com."),
]
for gold_ex, cited_ex in _sanity_pairs:
    assert _article_key(gold_ex) == _article_key(cited_ex), (
        f"Normaliseur KO : {gold_ex!r} ≠ {cited_ex!r} "
        f"({_article_key(gold_ex)!r} vs {_article_key(cited_ex)!r})"
    )

print("Fonctions utilitaires chargées (normaliseur article_recall : OK sur 4 cas).")


---
## M1 — Principe QA
**Format :** Question abstraite de droit → réponse avec articles + JP  
**Données :** 10 questions depuis Les-Audits-Affaires (ou fallback statique)  
**Métriques :** `keyword_recall`, `article_recall`

In [ ]:
# Chargement M1 : priorité au fichier complet, fallback sur le sample.
import random

m1_full_path   = Path(DATA_DIR) / "m1_full.json"
m1_sample_path = Path(DATA_DIR) / "m1_sample.json"

if m1_full_path.exists():
    m1_pool = json.loads(m1_full_path.read_text(encoding="utf-8"))
    source = m1_full_path
elif m1_sample_path.exists():
    m1_pool = json.loads(m1_sample_path.read_text(encoding="utf-8"))
    source = m1_sample_path
else:
    print(f"[WARN] Aucun fichier M1 dans {DATA_DIR}. Exécutez prepare_cluster_data.py en local.")
    m1_pool = []
    source = None

if m1_pool:
    n_target = min(M1_N_QUESTIONS, len(m1_pool))
    rng = random.Random(M1_SEED)
    m1_cases = rng.sample(m1_pool, n_target)
    print(f"M1 : {len(m1_pool)} questions dispo dans {source.name} "
          f"→ échantillon aléatoire de {len(m1_cases)} (seed={M1_SEED})")
else:
    m1_cases = []

# Aperçu : distribution par spécialisation
if m1_cases:
    df_preview = pd.DataFrame(m1_cases)
    print(f"\nRépartition par spécialisation :")
    print(df_preview["specialisation"].value_counts().to_string())
    print(f"\nPremières questions :")
    display(df_preview[["id", "specialisation", "question"]].head(5))


In [ ]:
M1_SYSTEM_PROMPT = """Tu es un juriste français expert. Tu réponds à des questions de droit positif français de manière FACTUELLE, PRÉCISE et NORMATIVE.

## RÈGLES IMPÉRATIVES

1. **Articles de loi** : tu DOIS citer au minimum 2 articles précis avec leur NUMÉRO.
   Formats acceptés (tu peux utiliser l'un ou l'autre) :
   - "article 1641 du Code civil"
   - "article L. 1121-1 du Code du travail"
   - "article L2141-2" (forme compacte)
   - "article 302 septies A"
   L'élément indispensable est le NUMÉRO de l'article (avec ou sans préfixe "L."). Sans numéro, la citation est invalide.
   INTERDIT : "le Code du travail", "les articles applicables", "la réglementation en vigueur".

2. **Jurisprudence** : tu DOIS citer au minimum 1 décision, au format :
   - "Cass. soc., 10 juill. 2002, n° 00-45.135"
   - "CE, 3 févr. 2016, n° 381828"
   - "Cass. com., 6 oct. 2021, n° 19-25.656"

3. **Pas de disclaimer défensif** : interdiction absolue de répondre "je ne peux pas donner de conseil personnalisé", "consultez un avocat", "cela dépend de votre situation". Tu ES le juriste expert, tu exposes la règle de droit applicable. L'utilisateur sait qu'il devra valider avec son propre avocat.

4. **Concision** : 3-5 paragraphes maximum. Structure : règle → exceptions → sanctions. Va droit au but.

5. **Cohérence JSON** : la liste `articles_cites` doit contenir EXACTEMENT les articles cités dans la réponse, un par entrée, DANS LA MÊME FORME que dans le texte. Idem pour `jurisprudence_citee`. Exemple : si tu cites "article L. 1121-1 du Code du travail" dans la réponse, tu mets "article L. 1121-1 du Code du travail" dans la liste.

6. **Certitude** : `niveau_certitude` = "élevé" si règle positive et stable, "moyen" si interprétation jurisprudentielle divisée, "faible" si question très controversée ou récente.

Réponds UNIQUEMENT avec le JSON demandé, sans préambule ni postambule."""


def build_m1_user_prompt(case: dict) -> str:
    return f"""Question juridique à traiter (schéma M1Output) :

QUESTION : {case['question']}

Domaine : {case.get('specialisation', 'non précisé')}

Rappel : minimum 2 articles AVEC NUMÉRO + 1 arrêt. Pas de refus de répondre."""


print("Prompts M1 définis (v2 : format article aligné avec le gold via normaliseur).")


In [ ]:
m1_results = []

for case in tqdm(m1_cases, desc="M1 — Principe QA"):
    output, meta = call_llm_structured(
        system_prompt=M1_SYSTEM_PROMPT,
        user_prompt=build_m1_user_prompt(case),
        output_model=M1Output,
        max_tokens=MAX_TOKENS_M1,
    )

    # Schéma normalisé : mêmes colonnes pour succès ET erreurs
    # (évite les CSV avec colonnes manquantes quand certaines lignes échouent)
    row = {
        "id":                   case["id"],
        "specialisation":       case.get("specialisation", ""),
        "question":             case["question"],
        "gold_articles":        case.get("gold_articles", []),
        "gold_keywords":        case.get("gold_keywords", []),
        "reponse":              None,
        "articles_cites":       [],
        "jurisprudence_citee":  [],
        "niveau_certitude":     None,
        "n_articles_cites":     0,
        "n_jp_citee":           0,
        "keyword_recall":       None,
        "article_recall":       None,
        "score_m1":             None,
        "latency_s":            meta["latency_s"],
        "tokens_used":          meta["tokens_used"],
        "finish_reason":        meta["finish_reason"],
        "error":                meta["error"],
        "raw_response":         meta.get("raw_response"),  # toujours conservé — diagnostic
    }

    if output is not None:
        kw_rec  = keyword_recall(output.reponse, case.get("gold_keywords", []))
        art_rec = article_recall(output.articles_cites, case.get("gold_articles", []))
        row.update({
            "reponse":             output.reponse,
            "articles_cites":      output.articles_cites,
            "jurisprudence_citee": output.jurisprudence_citee,
            "niveau_certitude":    output.niveau_certitude,
            "n_articles_cites":    len(output.articles_cites),
            "n_jp_citee":          len(output.jurisprudence_citee),
            "keyword_recall":      round(kw_rec, 3),
            "article_recall":      round(art_rec, 3),
            "score_m1":            round((kw_rec + art_rec) / 2, 3),
        })

    m1_results.append(row)

df_m1 = pd.DataFrame(m1_results)
n_ok  = df_m1["error"].isna().sum()
n_err = len(df_m1) - n_ok
print(f"M1 terminé — {len(df_m1)} réponses  (✓ {n_ok} succès · ✗ {n_err} erreurs)")


In [ ]:
# Affichage résultats M1 — adapté à la taille de l'échantillon
print("═" * 80)
print(f"RÉSULTATS M1 — Principe QA  ({len(df_m1)} questions)")
print("═" * 80)

display_cols = [c for c in [
    "id", "specialisation", "keyword_recall", "article_recall", "score_m1",
    "niveau_certitude", "n_articles_cites", "n_jp_citee", "latency_s", "error",
] if c in df_m1.columns]

# À >20 cas on n'affiche qu'un extrait (le CSV a le détail complet)
if len(df_m1) > 20:
    print(f"\nPremières lignes (20/{len(df_m1)}) :")
    print(df_m1[display_cols].head(20).to_string(index=False))
    print(f"\n... +{len(df_m1)-20} autres (voir le CSV pour le détail complet)")
else:
    print(df_m1[display_cols].to_string(index=False))

# ── Métriques agrégées (sur les lignes valides uniquement) ────────────
df_ok = df_m1[df_m1["error"].isna()]
if len(df_ok) > 0:
    print("\n" + "─" * 80)
    print(f"Métriques agrégées — {len(df_ok)} réponses valides / {len(df_m1)} totales")
    print("─" * 80)
    print(f"Score M1 moyen          : {df_ok['score_m1'].mean():.3f}  (σ = {df_ok['score_m1'].std():.3f})")
    print(f"Keyword recall (moy)    : {df_ok['keyword_recall'].mean():.3f}")
    print(f"Article recall (moy)    : {df_ok['article_recall'].mean():.3f}")
    print(f"N articles cités (moy)  : {df_ok['n_articles_cites'].mean():.1f}")
    print(f"N JP citées (moy)       : {df_ok['n_jp_citee'].mean():.1f}")
    print(f"Latence moyenne         : {df_ok['latency_s'].mean():.2f}s  (total : {df_ok['latency_s'].sum():.1f}s)")

    # Distribution niveau_certitude
    print("\n▸ Distribution niveau_certitude :")
    print(df_ok["niveau_certitude"].value_counts().to_string())

    # Score par spécialisation (utile à N=500)
    if df_ok["specialisation"].nunique() > 1:
        print("\n▸ Score par spécialisation :")
        by_spec = df_ok.groupby("specialisation").agg(
            n=("id", "count"),
            score=("score_m1", "mean"),
            art_rec=("article_recall", "mean"),
            kw_rec=("keyword_recall", "mean"),
        ).round(3).sort_values("score", ascending=False)
        print(by_spec.to_string())

# ── Diagnostic des erreurs (utile à N=500) ────────────────────────────
if n_err := (len(df_m1) - len(df_ok)):
    print(f"\n▸ Diagnostic des {n_err} erreurs :")
    err_types = df_m1[df_m1["error"].notna()]["finish_reason"].value_counts()
    print(err_types.to_string())

# ── Détails complets : uniquement si échantillon petit ────────────────
if len(df_m1) <= 20:
    print_full_details(
        df_m1,
        sections=[
            ("Question",                   "question"),
            ("Articles gold",              "gold_articles"),
            ("Articles cités",             "articles_cites"),
            ("JP citée",                   "jurisprudence_citee"),
            ("Niveau certitude",           "niveau_certitude"),
            ("Réponse LLM",                "reponse"),
            ("Erreur",                     "error"),
            ("Réponse brute (si erreur)",  "raw_response"),
        ],
        title="M1 — Détails complets",
    )


---
## M2 — Applied QA (avec contexte client)
**Format :** Dossier client + question → analyse contextualisée (articles + risques + stratégie)  
**Données :** 3 cas construits manuellement  
**Métriques :** `article_recall`, `strategy_present`

In [ ]:
# ── Cas M2 (hardcodés) ───────────────────────────────────────────────

M2_CASES = [
    {
        "id": "M2-S-001",
        "specialisation": "Droit Social",
        "difficulty": "easy",
        "case_summary": (
            "Salarié cadre avec 5 ans d'ancienneté. Son contrat comprend une clause de non-concurrence "
            "valable 2 ans sur l'ensemble du territoire national, sans aucune contrepartie financière mentionnée. "
            "Il vient de démissionner et veut rejoindre un concurrent direct dans la même ville."
        ),
        "client_position": "defense",
        "key_facts": [
            "clause de non-concurrence dans le contrat",
            "aucune contrepartie financière prévue",
            "périmètre national (territoire entier)",
            "durée 2 ans",
            "poste visé = concurrent direct",
        ],
        "question": "L'ancien employeur peut-il contraindre le salarié à respecter cette clause de non-concurrence ?",
        "gold_articles": ["L. 1121-1 Code du travail"],
        "gold_keywords": ["contrepartie", "nulle", "non-concurrence", "invalide"],
    },
    {
        "id": "M2-S-002",
        "specialisation": "Droit Civil",
        "difficulty": "medium",
        "case_summary": (
            "Particulier a acheté il y a 8 mois une voiture d'occasion auprès d'un vendeur professionnel. "
            "La boîte automatique vient de lâcher (coût de réparation : 4 500 €). Le garagiste confirme "
            "que le vice était antérieur à la vente (usure anormale interne invisible à l'achat). "
            "Le vendeur refuse tout remboursement et invoque l'état de la voiture au moment de la vente."
        ),
        "client_position": "demande",
        "key_facts": [
            "achat il y a 8 mois (dans le délai de prescription)",
            "vice antérieur à la vente (attestation garagiste)",
            "vendeur professionnel (présomption de connaissance du vice)",
            "vice invisible à l'inspection normale",
            "préjudice : 4 500 €",
        ],
        "question": "Quels sont les recours de l'acheteur contre le vendeur professionnel ?",
        "gold_articles": ["1641 Code civil", "1643 Code civil", "1644 Code civil"],
        "gold_keywords": ["vice caché", "action rédhibitoire", "estimatoire", "dommages-intérêts", "professionnel"],
    },
    {
        "id": "M2-S-003",
        "specialisation": "Droit Pénal",
        "difficulty": "hard",
        "case_summary": (
            "Commerçant qui a surpris un individu en train de voler dans son magasin. "
            "Il l'a physiquement retenu (bras dans le dos, pendant 12 minutes) "
            "jusqu'à l'arrivée de la police. L'individu s'est légèrement blessé au poignet. "
            "Il porte plainte pour séquestration et violences. Le commerçant est poursuivi."
        ),
        "client_position": "defense",
        "key_facts": [
            "flagrant délit de vol constaté",
            "rétention physique brève (12 minutes, jusqu'à la police)",
            "blessure légère (poignet)",
            "plainte double : séquestration + violences",
        ],
        "question": (
            "Le commerçant peut-il invoquer une cause d'irresponsabilité pour la rétention physique "
            "et les blessures légères occasionnées ?"
        ),
        "gold_articles": ["122-5 Code pénal", "122-7 Code pénal", "73 Code de procédure pénale"],
        "gold_keywords": ["légitime défense", "état de nécessité", "citoyen", "appréhension", "flagrant délit"],
    },
]

print(f"M2 : {len(M2_CASES)} cas chargés")
for c in M2_CASES:
    print(f"  {c['id']} — {c['specialisation']} ({c['difficulty']})")

In [ ]:
M2_SYSTEM_PROMPT = """Tu es un juriste français expert spécialisé en conseil client.
Tu analyses des situations juridiques précises et fournis des réponses opérationnelles.
Tes réponses incluent les textes applicables, les risques et une stratégie concrète.
Réponds UNIQUEMENT avec le JSON demandé, sans texte avant ou après."""


def build_m2_user_prompt(case: dict) -> str:
    facts = "\n".join(f"  - {f}" for f in case["key_facts"])
    pos = "demandeur" if case["client_position"] == "demande" else "défendeur"
    return f"""Analyse la situation juridique suivante (JSON schéma M2Output) :

DOMAINE : {case['specialisation']}
POSITION CLIENT : {pos}

RÉSUMÉ DU DOSSIER :
{case['case_summary']}

FAITS CLÉS :
{facts}

QUESTION : {case['question']}"""


m2_results = []

for case in tqdm(M2_CASES, desc="M2 — Applied QA"):
    output, meta = call_llm_structured(
        system_prompt=M2_SYSTEM_PROMPT,
        user_prompt=build_m2_user_prompt(case),
        output_model=M2Output,
        max_tokens=MAX_TOKENS_M2,
    )

    # Schéma normalisé : mêmes colonnes succès ET échec
    row = {
        "id":                   case["id"],
        "specialisation":       case["specialisation"],
        "difficulty":           case["difficulty"],
        "question":             case["question"],
        "case_summary":         case["case_summary"],
        "key_facts":            case["key_facts"],
        "gold_articles":        case.get("gold_articles", []),
        "gold_keywords":        case.get("gold_keywords", []),
        "reponse":              None,
        "articles_cites":       [],
        "jurisprudence_citee":  [],
        "risques":              None,
        "strategie":            None,
        "keyword_recall":       None,
        "article_recall":       None,
        "strategy_present":     None,
        "score_m2":             None,
        "latency_s":            meta["latency_s"],
        "tokens_used":          meta["tokens_used"],
        "finish_reason":        meta["finish_reason"],
        "error":                meta["error"],
        "raw_response":         meta.get("raw_response"),
    }

    if output is not None:
        art_rec = article_recall(output.articles_cites, case.get("gold_articles", []))
        kw_rec  = keyword_recall(output.reponse, case.get("gold_keywords", []))
        has_strategy = len(output.strategie) > 20
        row.update({
            "reponse":             output.reponse,
            "articles_cites":      output.articles_cites,
            "jurisprudence_citee": output.jurisprudence_citee,
            "risques":             output.risques,
            "strategie":           output.strategie,
            "keyword_recall":      round(kw_rec, 3),
            "article_recall":      round(art_rec, 3),
            "strategy_present":    has_strategy,
            "score_m2":            round((kw_rec + art_rec + int(has_strategy)) / 3, 3),
        })
    m2_results.append(row)

df_m2 = pd.DataFrame(m2_results)
n_ok  = df_m2["error"].isna().sum()
print(f"M2 terminé — {len(df_m2)} réponses  (✓ {n_ok} succès · ✗ {len(df_m2)-n_ok} erreurs)")

# ── Table compacte (métriques) ───────────────────────────────────────
print("\n═══ RÉSULTATS M2 (métriques) ═══")
display_cols = [c for c in ["id", "specialisation", "difficulty", "keyword_recall", "article_recall", "strategy_present", "score_m2", "latency_s", "error"] if c in df_m2.columns]
print(df_m2[display_cols].to_string(index=False))
df_m2_ok = df_m2[df_m2["error"].isna()]
if len(df_m2_ok) > 0:
    print(f"\nScore M2 moyen : {df_m2_ok['score_m2'].mean():.3f}")

# ── Détails complets ──────────────────────────────────────────────────
print_full_details(
    df_m2,
    sections=[
        ("Résumé dossier",              "case_summary"),
        ("Faits clés",                  "key_facts"),
        ("Question",                    "question"),
        ("Articles gold",               "gold_articles"),
        ("Articles cités",              "articles_cites"),
        ("JP citée",                    "jurisprudence_citee"),
        ("Réponse LLM",                 "reponse"),
        ("Risques",                     "risques"),
        ("Stratégie",                   "strategie"),
        ("Erreur",                      "error"),
        ("Réponse brute (si erreur)",   "raw_response"),
    ],
    title="M2 — Détails complets",
)


---
## M3 — Raisonnement multi-saut ⛔ SKIP
**Raison :** Nécessite le Knowledge Graph v1 (citations entre décisions) — non construit à ce stade.  
**Prévu :** Phase B3 du plan de construction.

In [ ]:
m3_status = {"module": "M3", "status": "SKIP", "reason": "KG v1 non disponible", "score": None}
print(f"M3 — Raisonnement multi-saut : SKIP ({m3_status['reason']})")

---
## M4 — QCM Retrieval
**Format :** Question à choix multiples (A/B/C/D) — 5 spécialisations couvertes  
**Données :** 5 QCM construits manuellement depuis le droit positif  
**Métriques :** `accuracy` (simple, automatique)

In [ ]:
M4_QCM = [
    {
        "id": "M4-QCM-001",
        "specialisation": "Droit Social",
        "difficulty": "easy",
        "question": "La clause de non-concurrence dans un contrat de travail est valide si elle comporte OBLIGATOIREMENT :",
        "options": {
            "A": "Une limite géographique uniquement",
            "B": "Une limite dans le temps, dans l'espace, une contrepartie financière et la protection d'intérêts légitimes",
            "C": "Une limite temporelle uniquement",
            "D": "L'accord écrit du salarié au moment de la rupture",
        },
        "gold_answer": "B",
        "gold_article": "Cass. soc., 10 juill. 2002 / L. 1121-1 C. trav.",
    },
    {
        "id": "M4-QCM-002",
        "specialisation": "Droit Civil",
        "difficulty": "easy",
        "question": "En matière de vices cachés (art. 1641 C. civ.), l'action de l'acheteur se prescrit en :",
        "options": {
            "A": "1 an à compter de la découverte",
            "B": "2 ans à compter de la découverte du vice",
            "C": "5 ans à compter de la vente",
            "D": "10 ans à compter de la livraison du bien",
        },
        "gold_answer": "B",
        "gold_article": "1648 Code civil",
    },
    {
        "id": "M4-QCM-003",
        "specialisation": "Droit Pénal",
        "difficulty": "medium",
        "question": "La légitime défense (art. 122-5 C. pén.) suppose TOUJOURS que la riposte soit :",
        "options": {
            "A": "Immédiate et préméditée contre l'agresseur identifié",
            "B": "Nécessaire, simultanée et proportionnée à l'agression réelle ou imminente",
            "C": "Exercée uniquement par les forces de l'ordre",
            "D": "Autorisée uniquement en cas d'agression armée",
        },
        "gold_answer": "B",
        "gold_article": "122-5 Code pénal",
    },
    {
        "id": "M4-QCM-004",
        "specialisation": "Droit Commercial",
        "difficulty": "medium",
        "question": "La rupture brutale des relations commerciales établies (art. L. 442-1, II C. com.) suppose :",
        "options": {
            "A": "Obligatoirement l'existence d'un contrat écrit entre les parties",
            "B": "Une rupture sans préavis écrit raisonnable eu égard à la durée de la relation",
            "C": "Un chiffre d'affaires annuel minimum prouvé par factures",
            "D": "Une rupture dans le cadre d'une procédure collective de l'auteur",
        },
        "gold_answer": "B",
        "gold_article": "L. 442-1 II Code de commerce",
    },
    {
        "id": "M4-QCM-005",
        "specialisation": "Droit de la Famille",
        "difficulty": "easy",
        "question": "La prestation compensatoire (art. 270-271 C. civ.) est due :",
        "options": {
            "A": "Uniquement par le conjoint qui a initié la procédure de divorce",
            "B": "Par le conjoint le plus aisé, pour compenser la disparité des conditions de vie créée par la rupture",
            "C": "Uniquement dans le cadre du divorce pour faute",
            "D": "Par le conjoint qui a la garde exclusive des enfants",
        },
        "gold_answer": "B",
        "gold_article": "270-271 Code civil",
    },
]

print(f"M4 : {len(M4_QCM)} QCM chargés")
for q in M4_QCM:
    print(f"  {q['id']} — {q['specialisation']} | Gold: {q['gold_answer']}")

In [ ]:
M4_SYSTEM_PROMPT = """Tu es un juriste français expert. Tu réponds à des questions à choix multiples de droit français.
Tu choisis la SEULE réponse correcte parmi A, B, C ou D et tu justifies ton choix.
Réponds UNIQUEMENT avec le JSON demandé, sans texte avant ou après."""


def build_m4_user_prompt(qcm: dict) -> str:
    opts = "\n".join(f"{k}) {v}" for k, v in qcm["options"].items())
    return f"""Question à choix multiples (JSON schéma M4Output) :

DOMAINE : {qcm['specialisation']}

QUESTION : {qcm['question']}

{opts}"""


m4_results = []

for qcm in tqdm(M4_QCM, desc="M4 — QCM"):
    output, meta = call_llm_structured(
        system_prompt=M4_SYSTEM_PROMPT,
        user_prompt=build_m4_user_prompt(qcm),
        output_model=M4Output,
        max_tokens=MAX_TOKENS_M4,
    )

    options_str = "\n".join(f"  {k}) {v}" for k, v in qcm["options"].items())

    # Schéma normalisé : mêmes colonnes succès ET échec
    row = {
        "id":             qcm["id"],
        "specialisation": qcm["specialisation"],
        "difficulty":     qcm["difficulty"],
        "question":       qcm["question"],
        "options":        options_str,
        "gold":           qcm["gold_answer"],
        "gold_article":   qcm.get("gold_article", ""),
        "predicted":      None,
        "justification":  None,
        "correct":        None,
        "score_m4":       None,
        "latency_s":      meta["latency_s"],
        "tokens_used":    meta["tokens_used"],
        "finish_reason":  meta["finish_reason"],
        "error":          meta["error"],
        "raw_response":   meta.get("raw_response"),
    }

    if output is not None:
        correct = output.reponse == qcm["gold_answer"]
        row.update({
            "predicted":     output.reponse,
            "justification": output.justification,
            "correct":       correct,
            "score_m4":      1.0 if correct else 0.0,
        })
    m4_results.append(row)

df_m4 = pd.DataFrame(m4_results)
n_ok  = df_m4["error"].isna().sum()
print(f"M4 terminé — {len(df_m4)} réponses  (✓ {n_ok} succès · ✗ {len(df_m4)-n_ok} erreurs)")

# ── Table compacte (métriques) ───────────────────────────────────────
print("\n═══ RÉSULTATS M4 (métriques) ═══")
display_cols = [c for c in ["id", "specialisation", "predicted", "gold", "correct", "latency_s", "error"] if c in df_m4.columns]
print(df_m4[display_cols].to_string(index=False))
df_m4_ok = df_m4[df_m4["error"].isna()]
if len(df_m4_ok) > 0:
    acc = df_m4_ok["correct"].mean()
    print(f"\nAccuracy M4 : {acc:.1%} ({df_m4_ok['correct'].sum()}/{len(df_m4_ok)})")

# ── Détails complets ──────────────────────────────────────────────────
print_full_details(
    df_m4,
    sections=[
        ("Question",                    "question"),
        ("Options",                     "options"),
        ("Prédit",                      "predicted"),
        ("Gold",                        "gold"),
        ("Article gold",                "gold_article"),
        ("Correct",                     "correct"),
        ("Justification",               "justification"),
        ("Erreur",                      "error"),
        ("Réponse brute (si erreur)",   "raw_response"),
    ],
    title="M4 — Détails complets",
)


---
## M5 — Temporalité ⛔ SKIP
**Raison :** Nécessite un KG avec versioning temporel des articles — non construit à ce stade.  
**Prévu :** Phase B3 du plan de construction.

In [ ]:
m5_status = {"module": "M5", "status": "SKIP", "reason": "Versioning temporel des articles non disponible", "score": None}
print(f"M5 — Temporalité : SKIP ({m5_status['reason']})")

---
## M6 — Interprétation d'arrêt contextualisée ⭐
**Format :** Dossier client + décision fournie → analyse à 7 dimensions  
**Données :** 5 cas gold standard annotés manuellement  
**Métriques :** `is_favorable_acc`, `sens_arret_acc`, `relevance_mae`, `principles_overlap`  
**Cas piège :** M6-JP-004 (cassation procédurale — ne doit PAS être is_favorable=True)

In [ ]:
m6_path = Path(DATA_DIR) / "m6_gold_cases.json"

if m6_path.exists():
    m6_cases = json.loads(m6_path.read_text(encoding="utf-8"))
    print(f"M6 : {len(m6_cases)} cas gold chargés depuis {m6_path}")
    for c in m6_cases:
        fav = "✓ favorable" if c['gold']['is_favorable'] else "✗ défav."
        trap = " [PIÈGE]" if c['difficulty'] == 'trap' else ""
        print(f"  {c['id']} — {c['specialisation']} ({c['difficulty']}) | {c['gold']['sens_arret']} | {fav}{trap}")
else:
    print(f"[WARN] {m6_path} introuvable.")
    print("       Exécutez prepare_cluster_data.py en local puis transférez cluster_data/")
    m6_cases = []

In [ ]:
M6_SYSTEM_PROMPT = """Tu es un avocat expert en droit français. Tu analyses des décisions de justice \
dans le contexte d'un dossier client précis.
Tu évalues avec rigueur si la décision est utile pour le client, en distinguant :
- Ce que décide réellement la cour (fond vs procédure)
- Si la décision est vraiment favorable au client (attention aux cassations procédurales)
- Les principes de droit directement transférables au dossier
Réponds UNIQUEMENT avec le JSON demandé, sans texte avant ou après."""


def build_m6_user_prompt(case: dict) -> str:
    c = case["case"]
    facts = "\n".join(f"  - {f}" for f in c["key_facts"])
    pos = "demandeur" if c["client_position"] == "demande" else "défendeur"
    decision_text = format_decision_for_prompt(case["decision"])

    return f"""Analyse la décision de justice fournie pour le dossier client ci-dessous (JSON schéma M6Output) :

━━━ DOSSIER CLIENT ━━━
Spécialisation : {case['specialisation']}
Position client : {pos}

Résumé : {c['case_summary']}

Faits clés :
{facts}

Question : {case['question']}

━━━ DÉCISION DE JUSTICE ━━━
{decision_text}"""


m6_results = []

for case in tqdm(m6_cases, desc="M6 — Interprétation d'arrêt"):
    output, meta = call_llm_structured(
        system_prompt=M6_SYSTEM_PROMPT,
        user_prompt=build_m6_user_prompt(case),
        output_model=M6Output,
        max_tokens=MAX_TOKENS_M6,
    )

    gold = case["gold"]
    c = case["case"]
    decision_text = format_decision_for_prompt(case["decision"])

    # Schéma normalisé : mêmes colonnes succès ET échec
    row = {
        "id":                         case["id"],
        "specialisation":             case["specialisation"],
        "difficulty":                 case["difficulty"],
        # Dossier client
        "case_summary":               c["case_summary"],
        "key_facts":                  c["key_facts"],
        "question":                   case["question"],
        # Décision passée au LLM
        "decision_text":              decision_text,
        # Gold
        "gold_is_favorable":          gold["is_favorable"],
        "gold_sens_arret":            gold["sens_arret"],
        "gold_relevance":             gold["relevance"],
        "gold_principles":            gold.get("principles_extracted", []),
        # Prédictions (remplies si output OK)
        "pred_camp_in_decision":      None,
        "pred_is_favorable":          None,
        "pred_sens_arret":            None,
        "pred_relevance":             None,
        "pred_dispositif_summary":    None,
        "pred_principles_extracted":  [],
        "pred_transfer_reasoning":    None,
        # Métriques
        "is_favorable_ok":            None,
        "sens_arret_ok":              None,
        "relevance_mae":              None,
        "principles_overlap":         None,
        "score_m6":                   None,
        # Méta
        "latency_s":                  meta["latency_s"],
        "tokens_used":                meta["tokens_used"],
        "finish_reason":              meta["finish_reason"],
        "error":                      meta["error"],
        "raw_response":               meta.get("raw_response"),
    }

    if output is not None:
        is_fav_ok    = output.is_favorable == gold["is_favorable"]
        sens_ok      = output.sens_arret   == gold["sens_arret"]
        rel_mae      = abs(output.relevance - gold["relevance"])
        princ_olap   = principles_keyword_overlap(
            output.principles_extracted, gold["principles_extracted"]
        )
        # Score composite (is_favorable pondéré 2x car critique)
        composite = (2 * int(is_fav_ok) + int(sens_ok) + (1 - rel_mae) + princ_olap) / 5

        row.update({
            "pred_camp_in_decision":      output.camp_in_decision,
            "pred_is_favorable":          output.is_favorable,
            "pred_sens_arret":            output.sens_arret,
            "pred_relevance":             round(output.relevance, 2),
            "pred_dispositif_summary":    output.dispositif_summary,
            "pred_principles_extracted":  output.principles_extracted,
            "pred_transfer_reasoning":    output.transfer_reasoning,
            "is_favorable_ok":            is_fav_ok,
            "sens_arret_ok":              sens_ok,
            "relevance_mae":              round(rel_mae, 3),
            "principles_overlap":         round(princ_olap, 3),
            "score_m6":                   round(composite, 3),
        })
    m6_results.append(row)

df_m6 = pd.DataFrame(m6_results)
n_ok  = df_m6["error"].isna().sum()
print(f"M6 terminé — {len(df_m6)} réponses  (✓ {n_ok} succès · ✗ {len(df_m6)-n_ok} erreurs)")


In [ ]:
print("═" * 80)
print("RÉSULTATS M6 — Interprétation d'arrêt (métriques)")
print("═" * 80)

display_cols = [c for c in [
    "id", "difficulty",
    "gold_is_favorable", "pred_is_favorable", "is_favorable_ok",
    "gold_sens_arret",   "pred_sens_arret",   "sens_arret_ok",
    "gold_relevance",    "pred_relevance",    "relevance_mae",
    "principles_overlap", "score_m6", "latency_s"
] if c in df_m6.columns]

print(df_m6[display_cols].to_string(index=False))

if "score_m6" in df_m6.columns:
    print(f"""
Métriques M6 :
  is_favorable accuracy : {df_m6['is_favorable_ok'].mean():.1%}  (métrique cœur — piège sur M6-JP-004)
  sens_arret accuracy   : {df_m6['sens_arret_ok'].mean():.1%}
  relevance MAE         : {df_m6['relevance_mae'].mean():.3f}  (0=parfait)
  principles overlap    : {df_m6['principles_overlap'].mean():.3f}  (Jaccard kw)
  score M6 composite    : {df_m6['score_m6'].mean():.3f}
""")

    # Identifier si le modèle a bien géré le cas piège
    trap = df_m6[df_m6["difficulty"] == "trap"]
    if not trap.empty:
        trap_ok = trap["is_favorable_ok"].all()
        status = "✓ PIÈGE DÉTECTÉ" if trap_ok else "✗ PIÈGE RATÉ (LLM naïf)"
        print(f"  Cas piège (M6-JP-004) : {status}")

# ── Détails complets (dossier + décision + prédictions + gold) ──────
print_full_details(
    df_m6,
    sections=[
        ("Résumé dossier",        "case_summary"),
        ("Faits clés",            "key_facts"),
        ("Question",              "question"),
        ("Décision (fournie LLM)","decision_text"),
        # Gold
        ("Gold is_favorable",     "gold_is_favorable"),
        ("Gold sens_arret",       "gold_sens_arret"),
        ("Gold relevance",        "gold_relevance"),
        ("Gold principes",        "gold_principles"),
        # Prédictions LLM (plein)
        ("Pred camp",             "pred_camp_in_decision"),
        ("Pred is_favorable",     "pred_is_favorable"),
        ("Pred sens_arret",       "pred_sens_arret"),
        ("Pred relevance",        "pred_relevance"),
        ("Pred dispositif",       "pred_dispositif_summary"),
        ("Pred principes",        "pred_principles_extracted"),
        ("Pred transfer (raisonnement)", "pred_transfer_reasoning"),
        ("Erreur",                "error"),
    ],
    title="M6 — Détails complets",
)


---
## Résultats consolidés — B1 (LLM seul, zero-shot)
Agrégation des scores par module + export CSV

In [ ]:
# ── Tableau de synthèse ───────────────────────────────────────────────
summary_rows = []

if not df_m1.empty and "score_m1" in df_m1.columns:
    summary_rows.append({
        "module": "M1 — Principe QA",
        "n_items": len(df_m1),
        "main_metric": "score (kw+art recall) / 2",
        "score": round(df_m1["score_m1"].mean(), 3),
        "latency_mean_s": round(df_m1["latency_s"].mean(), 2),
        "status": "OK",
    })

if not df_m2.empty and "score_m2" in df_m2.columns:
    summary_rows.append({
        "module": "M2 — Applied QA",
        "n_items": len(df_m2),
        "main_metric": "score (kw+art+strategy) / 3",
        "score": round(df_m2["score_m2"].mean(), 3),
        "latency_mean_s": round(df_m2["latency_s"].mean(), 2),
        "status": "OK",
    })

summary_rows.append({"module": "M3 — Multi-saut", "n_items": 0, "main_metric": "—", "score": None, "latency_mean_s": None, "status": "SKIP (KG v1 manquant)"})

if not df_m4.empty and "score_m4" in df_m4.columns:
    summary_rows.append({
        "module": "M4 — QCM",
        "n_items": len(df_m4),
        "main_metric": "accuracy",
        "score": round(df_m4["score_m4"].mean(), 3),
        "latency_mean_s": round(df_m4["latency_s"].mean(), 2),
        "status": "OK",
    })

summary_rows.append({"module": "M5 — Temporalité", "n_items": 0, "main_metric": "—", "score": None, "latency_mean_s": None, "status": "SKIP (versioning temporel manquant)"})

if not df_m6.empty and "score_m6" in df_m6.columns:
    summary_rows.append({
        "module": "M6 — Interprétation d'arrêt ⭐",
        "n_items": len(df_m6),
        "main_metric": "composite (is_fav×2 + sens + (1-mae) + overlap) / 5",
        "score": round(df_m6["score_m6"].mean(), 3),
        "latency_mean_s": round(df_m6["latency_s"].mean(), 2),
        "status": "OK",
    })

df_summary = pd.DataFrame(summary_rows)

print("═" * 80)
print(f"BENCHMARK M1–M6 — Configuration B1 (LLM seul) — Modèle : {MODEL_ID}")
print(f"Run : {RUN_ID}")
print("═" * 80)
print(df_summary[["module", "n_items", "score", "latency_mean_s", "status"]].to_string(index=False))

# Score global (moyenne pondérée des modules disponibles, M6 pondéré ×2)
scores_available = [(r["score"], 2 if "M6" in r["module"] else 1) for r in summary_rows if r["score"] is not None]
if scores_available:
    total_w = sum(w for _, w in scores_available)
    weighted_score = sum(s * w for s, w in scores_available) / total_w
    print(f"\nScore global pondéré (M6 ×2) : {weighted_score:.3f}")
    print(f"(M1:×1, M2:×1, M4:×1, M6:×2 — M3/M5 exclus)")

In [ ]:
# ── Export CSV ────────────────────────────────────────────────────────
results_base = Path(RESULTS_DIR) / f"{RUN_ID}_{MODEL_ID.replace('/', '_')}"

exports = [
    (df_m1,      f"{results_base}_M1.csv"),
    (df_m2,      f"{results_base}_M2.csv"),
    (df_m4,      f"{results_base}_M4.csv"),
    (df_m6,      f"{results_base}_M6.csv"),
    (df_summary, f"{results_base}_summary.csv"),
]

for df, path in exports:
    if not df.empty:
        df.to_csv(path, index=False, sep=";", encoding="utf-8-sig")
        print(f"Exporté : {path}")

# Export JSON consolidé
consolidated = {
    "run_id": RUN_ID,
    "model_id": MODEL_ID,
    "config": "B1_zero_shot",
    "summary": df_summary.to_dict(orient="records"),
    "m1_detail": df_m1.to_dict(orient="records") if not df_m1.empty else [],
    "m2_detail": df_m2.to_dict(orient="records") if not df_m2.empty else [],
    "m4_detail": df_m4.to_dict(orient="records") if not df_m4.empty else [],
    "m6_detail": df_m6.to_dict(orient="records") if not df_m6.empty else [],
}
json_path = f"{results_base}_full.json"
Path(json_path).write_text(json.dumps(consolidated, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Exporté : {json_path}")

In [ ]:
if not MATPLOTLIB_OK:
    print("matplotlib non disponible — graphiques ignorés")
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(f"Benchmark M1–M6 — B1 zero-shot\nModèle : {MODEL_ID}", fontsize=12, fontweight="bold")

    # ── Graphique 1 : Scores par module ──────────────────────────────────
    ax = axes[0]
    scores_df = df_summary[df_summary["score"].notna()].copy()
    colors = ["#2196F3" if "M6" not in r else "#FF9800" for r in scores_df["module"]]
    bars = ax.barh(scores_df["module"].str.replace(" ⭐", ""), scores_df["score"], color=colors)
    ax.set_xlim(0, 1)
    ax.set_xlabel("Score")
    ax.set_title("Scores par module")
    for bar, score in zip(bars, scores_df["score"]):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                f"{score:.3f}", va="center", fontsize=9)
    ax.axvline(x=0.5, color="gray", linestyle="--", alpha=0.5, label="seuil 0.5")
    ax.legend(fontsize=8)

    # ── Graphique 2 : M6 — Dimensions détaillées ─────────────────────────
    ax = axes[1]
    if not df_m6.empty and "is_favorable_ok" in df_m6.columns:
        m6_metrics = {
            "is_favorable": df_m6["is_favorable_ok"].mean(),
            "sens_arret":   df_m6["sens_arret_ok"].mean(),
            "1 - rel_mae":  1 - df_m6["relevance_mae"].mean(),
            "principles":   df_m6["principles_overlap"].mean(),
        }
        metric_names = list(m6_metrics.keys())
        metric_vals  = list(m6_metrics.values())
        bar_colors = ["#4CAF50" if v >= 0.6 else "#F44336" if v < 0.4 else "#FF9800" for v in metric_vals]
        ax.bar(metric_names, metric_vals, color=bar_colors)
        ax.set_ylim(0, 1)
        ax.set_ylabel("Score")
        ax.set_title("M6 — 4 dimensions automatiques")
        for i, v in enumerate(metric_vals):
            ax.text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=10, fontweight="bold")
        ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5)
    else:
        ax.text(0.5, 0.5, "Données M6\nnon disponibles", ha="center", va="center", transform=ax.transAxes)

    # ── Graphique 3 : M6 — is_favorable par cas (piège mis en évidence) ──
    ax = axes[2]
    if not df_m6.empty and "gold_is_favorable" in df_m6.columns:
        x = range(len(df_m6))
        gold_vals = df_m6["gold_is_favorable"].astype(int)
        pred_vals = df_m6["pred_is_favorable"].astype(int)
        labels    = df_m6["id"].str.replace("M6-JP-", "")
        colors_gold = ["#4CAF50" if v else "#F44336" for v in df_m6["gold_is_favorable"]]
        colors_pred = ["#4CAF50" if v else "#F44336" for v in df_m6["pred_is_favorable"]]

        ax.scatter([i - 0.15 for i in x], gold_vals, color=colors_gold, s=120, marker="o", label="Gold", zorder=3)
        ax.scatter([i + 0.15 for i in x], pred_vals, color=colors_pred, s=120, marker="^", label="Prédit", zorder=3)
        ax.set_xticks(list(x))
        ax.set_xticklabels(labels, rotation=30, fontsize=8)
        ax.set_yticks([0, 1])
        ax.set_yticklabels(["Défav.", "Fav."])
        ax.set_title("M6 — is_favorable\n(◯ gold, △ prédit)")
        ax.legend(fontsize=8)
        ax.set_ylim(-0.3, 1.3)
        # Surligner le cas piège
        trap_idx = df_m6[df_m6["difficulty"] == "trap"].index
        for idx in trap_idx:
            ax.axvspan(idx - 0.4, idx + 0.4, alpha=0.1, color="red")
            ax.text(idx, 1.2, "PIÈGE", ha="center", fontsize=7, color="red")
    else:
        ax.text(0.5, 0.5, "Données M6\nnon disponibles", ha="center", va="center", transform=ax.transAxes)

    plt.tight_layout()
    plot_path = f"{results_base}_plot.png"
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Graphique enregistré : {plot_path}")

---
## Interprétation des résultats

### Ce que ces scores mesurent
- **M1 / M2** : rappel lexical (articles, mots-clés) — métrique proxy, imparfaite (ne mesure pas la justesse réelle)
- **M4** : accuracy exacte — métrique fiable, format QCM
- **M6** : métriques automatiques sur 4 dimensions ; la 5e (`dispositif_summary`) et la 7e (`transfer_reasoning`) nécessitent un LLM-judge (prochaine itération)

### Cas piège M6-JP-004
- Gold : `is_favorable=False`, `relevance=0.20` (cassation purement procédurale)
- Un LLM naïf voit « cassation » → répond `is_favorable=True` → score 0 sur la métrique cœur
- **Si le modèle passe ce test → il comprend la distinction fond/procédure**

### Prochaines étapes
1. Ajouter LLM-as-judge (Judge 1 + Judge 2) pour `dispositif_summary` et `transfer_reasoning`
2. Tester configurations B2 (RAG vectoriel), B4 (RAG + agentique)
3. Comparer avec configuration C (GraphRAG)
4. Étendre à 30-40 cas par spécialisation pour M6 (target final)